# 🎓 Projet de Fin de Module – EMSI 2025-2026
## Apprentissage Automatique & Prétraitement des Données

| | |
|---|---|
| **Dataset 1** | Titanic (données tabulaires) |
| **Dataset 2** | 20 Newsgroups – 4 catégories (données textuelles) |
| **Auteur** | *[oussama banani]* |
| **Promotion** | 2025-2026 |

---


## 0. Imports & Configuration globale

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer, HashingVectorizer
)
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)

np.random.seed(42)

# Palette de couleurs cohérente dans tout le notebook
PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

print("✅ Imports OK – environnement prêt.")


---
# PARTIE I – Données Tabulaires : Titanic 🚢

> **Objectif :** Construire une chaîne complète de prétraitement et de prédiction
> sur un dataset tabulaire réel contenant des variables numériques, catégorielles
> et des valeurs manquantes.


## I.1 – Chargement du jeu de données

In [ ]:
# Chargement depuis GitHub (source officielle Kaggle)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df_raw = pd.read_csv(url)
    print(f"✅ Données chargées depuis l'URL distante.")
except Exception:
    # Fallback : reconstruction si hors-ligne
    print("⚠️  URL inaccessible – reconstruction d'un échantillon représentatif.")
    n = 891
    np.random.seed(42)
    survived = np.random.binomial(1, 0.38, n)
    pclass   = np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
    sex      = np.random.choice(["male", "female"], n, p=[0.65, 0.35])
    age      = np.where(np.random.normal(29, 13, n) < 0.5, np.nan,
                        np.random.normal(29, 13, n))
    age[np.random.choice(n, int(0.197*n), replace=False)] = np.nan
    fare     = np.where(pclass==1, np.random.exponential(80,n),
               np.where(pclass==2, np.random.exponential(20,n),
                        np.random.exponential(10,n)))
    embarked = np.random.choice(["S","C","Q",np.nan], n, p=[0.72,0.19,0.086,0.004])
    sibsp    = np.random.choice([0,1,2,3,4,5], n, p=[0.68,0.23,0.06,0.02,0.005,0.005])
    parch    = np.random.choice([0,1,2,3,4], n, p=[0.76,0.13,0.09,0.01,0.01])
    cabin_vals = [""] * n
    for i in range(n):
        if pclass[i] == 1 and np.random.rand() < 0.6:
            cabin_vals[i] = np.random.choice(["C85","B28","C23","E46","G6"])
    df_raw = pd.DataFrame({
        "PassengerId": range(1,n+1), "Survived": survived, "Pclass": pclass,
        "Name": [f"Doe, Mr. John {i}" for i in range(n)], "Sex": sex,
        "Age": age, "SibSp": sibsp, "Parch": parch,
        "Ticket": [f"T{np.random.randint(10000,99999)}" for _ in range(n)],
        "Fare": fare, "Cabin": cabin_vals, "Embarked": embarked,
    })

print(f"\nDimensions : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
df_raw.head()


## I.2 – Inspection initiale & lecture critique

In [ ]:
# Distribution de la variable cible
print("=== Variable cible : Survived ===")
vc = df_raw["Survived"].value_counts()
print(f"  Décédés (0)    : {vc.get(0,0):>4}  ({vc.get(0,0)/len(df_raw)*100:.1f} %)")
print(f"  Survivants (1) : {vc.get(1,0):>4}  ({vc.get(1,0)/len(df_raw)*100:.1f} %)")

# Types de variables
print("\n=== Types de variables ===")
print(df_raw.dtypes.to_string())


In [ ]:
# Valeurs manquantes
missing     = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({"Manquants": missing, "Pourcentage (%)": missing_pct})
missing_df  = missing_df[missing_df["Manquants"] > 0].sort_values("Pourcentage (%)", ascending=False)

print("=== Valeurs manquantes ===")
print(missing_df.to_string())


In [ ]:
# ── Figure 1 : Valeurs manquantes + distribution cible
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

missing_df["Pourcentage (%)"].plot(kind="bar", ax=axes[0], color=PALETTE[0], edgecolor="white")
axes[0].set_title("Fig 1a – Taux de valeurs manquantes", fontweight="bold")
axes[0].set_ylabel("Taux (%)"); axes[0].tick_params(axis="x", rotation=30)

df_raw["Survived"].value_counts().plot(kind="bar", ax=axes[1],
    color=[PALETTE[2], PALETTE[0]], edgecolor="white")
axes[1].set_title("Fig 1b – Distribution de la variable cible", fontweight="bold")
axes[1].set_xticklabels(["Décédé (0)", "Survivant (1)"], rotation=0)
axes[1].set_ylabel("Effectif")

plt.suptitle("Inspection initiale du dataset Titanic", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 2 : Distributions des variables numériques
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, ["Age", "Fare", "SibSp"], PALETTE[:3]):
    ax.hist(df_raw[col].dropna(), bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(f"Distribution – {col}", fontweight="bold")
    ax.set_xlabel(col); ax.set_ylabel("Effectif")
plt.suptitle("Fig 2 – Distributions des variables numériques", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# ── Figure 3 : Taux de survie par sexe et par classe
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_raw.groupby("Sex")["Survived"].mean().plot(
    kind="bar", ax=axes[0], color=[PALETTE[1], PALETTE[0]], edgecolor="white")
axes[0].set_title("Taux de survie par sexe", fontweight="bold")
axes[0].set_ylabel("Taux de survie"); axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)

df_raw.groupby("Pclass")["Survived"].mean().plot(
    kind="bar", ax=axes[1], color=PALETTE[2:5], edgecolor="white")
axes[1].set_title("Taux de survie par classe (Pclass)", fontweight="bold")
axes[1].set_ylabel("Taux de survie"); axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("Fig 3 – Analyse exploratoire de la survie", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


### Classification des variables qualitatives

| Type | Variables | Encodage retenu |
|---|---|---|
| **Nominales** | Sex, Embarked, Title | OneHotEncoder |
| **Ordinales** | Pclass (1 > 2 > 3) | OrdinalEncoder |
| **Numériques** | Age, Fare, SibSp, Parch, FamilySize | StandardScaler après imputation |
| **Binaires** | IsAlone, HasCabin | Passthrough (déjà 0/1) |

**Difficultés identifiées :**
- `Age` : 20,7 % de valeurs manquantes → imputation par la **médiane** (robuste aux outliers)
- `Cabin` : 77,1 % → **binarisation** en `HasCabin` (0/1)
- `Fare` : distribution asymétrique (longue queue droite)
- `Ticket`, `Name`, `PassengerId` : non informatifs → exclus


## I.3 – Ingénierie des variables

In [ ]:
def engineer_features(df):
    """Feature engineering reproduisible sur train et test."""
    d = df.copy()
    # Extraction du titre depuis le nom
    d["Title"] = d["Name"].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
    d["Title"] = d["Title"].replace({
        "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
        "Lady": "Rare", "Countess": "Rare", "Capt": "Rare",
        "Col": "Rare", "Don": "Rare", "Dr": "Rare",
        "Major": "Rare", "Rev": "Rare", "Sir": "Rare",
        "Jonkheer": "Rare", "Dona": "Rare"
    })
    d["FamilySize"]  = d["SibSp"] + d["Parch"] + 1
    d["IsAlone"]     = (d["FamilySize"] == 1).astype(int)
    d["HasCabin"]    = d["Cabin"].notna().astype(int)   # binarisation de Cabin
    d["FarePerPers"] = d["Fare"] / d["FamilySize"]
    return d

df = engineer_features(df_raw)

# Groupes de colonnes
COLS_NUM = ["Age", "Fare", "FarePerPers", "FamilySize", "SibSp", "Parch"]
COLS_ORD = ["Pclass"]
COLS_NOM = ["Sex", "Embarked", "Title"]
COLS_BIN = ["IsAlone", "HasCabin"]
TARGET   = "Survived"

X = df[COLS_NUM + COLS_ORD + COLS_NOM + COLS_BIN]
y = df[TARGET]

print("✅ Feature engineering terminé.")
print(f"   Nouvelles variables : Title, FamilySize, IsAlone, HasCabin, FarePerPers")
print(f"   Shape final de X : {X.shape}")


## I.4 – Split train/test stratifié (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train : {X_train.shape[0]} exemples | Test : {X_test.shape[0]} exemples")
print(f"Taux de survie train : {y_train.mean():.3f} | test : {y_test.mean():.3f}  (stratification OK)")


## I.5 – Construction du ColumnTransformer et des Pipelines

In [ ]:
# ── Transformateurs par type de variable
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # médiane = robuste aux outliers
    ("scaler",  StandardScaler())
])

ord_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[[1, 2, 3]]))  # ordre Pclass respecté
])

nom_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# ── ColumnTransformer : assemblage
preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, COLS_NUM),
    ("ord", ord_transformer, COLS_ORD),
    ("nom", nom_transformer, COLS_NOM),
    ("bin", "passthrough",   COLS_BIN),   # déjà en 0/1
])

# ── Pipeline A : Régression Logistique (interprétable, rapide)
pipe_lr = Pipeline([("preprocess", preprocessor),
                    ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42))])

# ── Pipeline B : Random Forest (non-linéaire, robuste)
pipe_rf = Pipeline([("preprocess", preprocessor),
                    ("clf", RandomForestClassifier(n_estimators=200, random_state=42))])

# ── Pipeline C : Gradient Boosting (boosting, haute capacité)
pipe_gb = Pipeline([("preprocess", preprocessor),
                    ("clf", GradientBoostingClassifier(n_estimators=150, random_state=42))])

print("✅ Trois pipelines construits :")
print("   A – LogisticRegression")
print("   B – RandomForest (200 arbres)")
print("   C – GradientBoosting (150 estimateurs)")


## I.6 – Validation croisée stratifiée (5 folds)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_tab = {}

for name, pipe in [("LogReg", pipe_lr), ("RandomForest", pipe_rf), ("GradientBoosting", pipe_gb)]:
    acc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    f1_scores  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    results_tab[name] = {
        "acc_mean": acc_scores.mean(), "acc_std": acc_scores.std(),
        "f1_mean" : f1_scores.mean(),  "f1_std" : f1_scores.std()
    }
    print(f"{name:<20}  Accuracy = {acc_scores.mean():.4f} ± {acc_scores.std():.4f}"
          f"  |  F1 = {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")


In [ ]:
# ── Figure 4 : Comparaison CV
fig, ax = plt.subplots(figsize=(9, 4))
names = list(results_tab.keys())
means = [results_tab[n]["acc_mean"] for n in names]
stds  = [results_tab[n]["acc_std"]  for n in names]
bars  = ax.bar(names, means, yerr=stds, capsize=7, color=PALETTE[:3],
               edgecolor="white", width=0.5)
ax.set_ylabel("Accuracy (CV 5-folds)"); ax.set_ylim(0, 1)
ax.set_title("Fig 4 – Comparaison des pipelines (validation croisée)", fontweight="bold")
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, mean + 0.01, f"{mean:.3f}",
            ha="center", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()


## I.7 – Recherche d'hyperparamètres (GridSearchCV – RandomForest)

In [ ]:
param_grid_rf = {
    "clf__n_estimators"    : [100, 200],
    "clf__max_depth"       : [None, 10, 15],
    "clf__min_samples_split": [2, 5],
}

gs_rf = GridSearchCV(pipe_rf, param_grid_rf, cv=cv, scoring="accuracy", n_jobs=-1, verbose=0)
gs_rf.fit(X_train, y_train)

print(f"Meilleurs hyperparamètres : {gs_rf.best_params_}")
print(f"Meilleur score CV         : {gs_rf.best_score_:.4f}")
best_rf = gs_rf.best_estimator_


## I.8 – Évaluation finale sur le jeu de test

In [ ]:
pipe_lr.fit(X_train, y_train)
pipe_gb.fit(X_train, y_train)

models_eval = {
    "LogisticRegression"  : pipe_lr,
    "RandomForest (tuned)": best_rf,
    "GradientBoosting"    : pipe_gb,
}

test_results = {}
for name, model in models_eval.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    test_results[name] = {"acc": acc, "f1": f1, "y_pred": y_pred}
    print(f"\n── {name}")
    print(f"   Accuracy : {acc:.4f}   F1 : {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Décédé", "Survivant"]))


In [ ]:
# ── Figure 5 : Matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models_eval.items()):
    cm = confusion_matrix(y_test, test_results[name]["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Décédé","Survivant"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=9, fontweight="bold")
plt.suptitle("Fig 5 – Matrices de confusion (jeu de test)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# ── Figure 6 : Importance des variables (Random Forest)
nom_enc = best_rf.named_steps["preprocess"].named_transformers_["nom"].named_steps["encoder"]
feat_names_nom = list(nom_enc.get_feature_names_out(COLS_NOM))
all_feat_names = COLS_NUM + COLS_ORD + feat_names_nom + COLS_BIN

importances = best_rf.named_steps["clf"].feature_importances_
indices = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([all_feat_names[i] for i in reversed(indices)],
        [importances[i]    for i in reversed(indices)],
        color=PALETTE[0], edgecolor="white")
ax.set_title("Fig 6 – Importance des variables (RandomForest)", fontweight="bold")
ax.set_xlabel("Importance (Gini)")
plt.tight_layout(); plt.show()


### 📝 Synthèse Partie I

Le **ColumnTransformer** sépare explicitement les traitements selon la nature des variables :
- Variables **numériques** → imputation médiane + StandardScaler
- Variables **ordinales** (Pclass) → OrdinalEncoder (l'ordre 1 > 2 > 3 est préservé)
- Variables **nominales** (Sex, Embarked, Title) → OneHotEncoder

Le **Pipeline** garantit l'absence de *data leakage* : toutes les transformations sont apprises sur le train set uniquement et appliquées telles quelles sur le test set.

Le **RandomForest optimisé** constitue le meilleur compromis performance / robustesse / interprétabilité sur ce dataset.


---
# PARTIE II – Données Textuelles : 20 Newsgroups 📰

> **Objectif :** Construire et comparer des représentations vectorielles de textes,
> puis classifier des articles de presse dans 4 catégories thématiques.


## II.1 – Chargement du corpus & présentation

In [ ]:
CATEGORIES = ["sci.med", "sci.space", "rec.sport.baseball", "talk.politics.guns"]

try:
    news_train = fetch_20newsgroups(
        subset="train", categories=CATEGORIES,
        remove=("headers", "footers", "quotes"), random_state=42)
    news_test = fetch_20newsgroups(
        subset="test", categories=CATEGORIES,
        remove=("headers", "footers", "quotes"), random_state=42)
    print("✅ Corpus 20 Newsgroups chargé depuis sklearn.")
    REAL_DATA = True
except Exception as e:
    print(f"⚠️  Corpus non disponible ({e}).")
    print("   → Génération d'un corpus synthétique illustratif.")
    REAL_DATA = False

if not REAL_DATA:
    np.random.seed(42)
    vocab = {
        0: ["patient","doctor","disease","treatment","hospital","medicine",
            "clinical","diagnosis","therapy","drug","symptoms","surgery"],
        1: ["orbit","nasa","shuttle","planet","satellite","mission",
            "launch","telescope","astronaut","cosmos","galaxy","probe"],
        2: ["baseball","pitcher","batter","innings","team","game",
            "score","league","season","runs","hit","stadium"],
        3: ["gun","weapon","rights","law","government","amendment",
            "firearms","control","police","armed","crime","legislation"],
    }
    labels_train, texts_train, labels_test, texts_test = [], [], [], []
    for label, words in vocab.items():
        for _ in range(600):
            texts_train.append(" ".join(np.random.choice(words, np.random.randint(60,150))))
            labels_train.append(label)
        for _ in range(200):
            texts_test.append(" ".join(np.random.choice(words, np.random.randint(60,150))))
            labels_test.append(label)

    class FakeNews:
        def __init__(self, data, target, target_names):
            self.data, self.target = data, np.array(target)
            self.target_names = target_names

    TN = ["sci.med","sci.space","rec.sport.baseball","talk.politics.guns"]
    news_train = FakeNews(texts_train, labels_train, TN)
    news_test  = FakeNews(texts_test,  labels_test,  TN)

X_text_train, y_text_train = news_train.data, news_train.target
X_text_test,  y_text_test  = news_test.data,  news_test.target
CLASS_NAMES = news_train.target_names

print(f"\nTrain : {len(X_text_train)} docs | Test : {len(X_text_test)} docs")
print(f"Classes : {CLASS_NAMES}")


In [ ]:
# Distribution des classes + longueur des docs
unique, counts = np.unique(y_text_train, return_counts=True)
doc_lengths = [len(doc.split()) for doc in X_text_train]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(CLASS_NAMES, counts, color=PALETTE[:4], edgecolor="white")
axes[0].set_title("Fig 7 – Distribution des classes (train)", fontweight="bold")
axes[0].set_ylabel("Nombre de documents"); axes[0].tick_params(axis="x", rotation=12)

axes[1].hist(doc_lengths, bins=50, color=PALETTE[1], edgecolor="white", alpha=0.85)
axes[1].set_title("Fig 8 – Longueur des documents (mots)", fontweight="bold")
axes[1].set_xlabel("Nombre de mots"); axes[1].set_ylabel("Effectif")

plt.suptitle("Exploration du corpus 20 Newsgroups", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Longueur moy : {np.mean(doc_lengths):.0f} mots | Médiane : {np.median(doc_lengths):.0f}")


## II.2 – Construction et comparaison des représentations textuelles

| Représentation | Description | Avantage | Limite |
|---|---|---|---|
| **CountVectorizer** | Comptage brut des occurrences | Simple, intuitif | Favorise les mots fréquents non discriminants |
| **TF-IDF (1-gram)** | Pondération fréquence × rareté documentaire | Discriminant | Ignore l'ordre des mots |
| **TF-IDF (1,2-gram)** | Unigrammes + bigrammes | Capture le contexte local | Dimensionnalité accrue |
| **HashingVectorizer** | Hachage sans dictionnaire | Très rapide, mémoire fixe | Irréversible, pas d'interprétation |


In [ ]:
COMMON_PARAMS = dict(stop_words="english", min_df=2, max_df=0.95, max_features=20000)

# 1. CountVectorizer
cv_count         = CountVectorizer(**COMMON_PARAMS)
X_train_count    = cv_count.fit_transform(X_text_train)
X_test_count     = cv_count.transform(X_text_test)

# 2. TF-IDF unigrammes
cv_tfidf         = TfidfVectorizer(**COMMON_PARAMS)
X_train_tfidf    = cv_tfidf.fit_transform(X_text_train)
X_test_tfidf     = cv_tfidf.transform(X_text_test)

# 3. TF-IDF bigrammes
cv_tfidf_bi      = TfidfVectorizer(**COMMON_PARAMS, ngram_range=(1,2))
X_train_tfidf_bi = cv_tfidf_bi.fit_transform(X_text_train)
X_test_tfidf_bi  = cv_tfidf_bi.transform(X_text_test)

# 4. HashingVectorizer
cv_hash          = HashingVectorizer(stop_words="english", n_features=2**16, alternate_sign=False)
X_train_hash     = cv_hash.transform(X_text_train)

def sparsity(X):
    return (1 - X.nnz / (X.shape[0] * X.shape[1])) * 100

print(f"CountVectorizer   : {X_train_count.shape}   sparsité = {sparsity(X_train_count):.1f} %")
print(f"TF-IDF (1-gram)   : {X_train_tfidf.shape}   sparsité = {sparsity(X_train_tfidf):.1f} %")
print(f"TF-IDF (1,2-gram) : {X_train_tfidf_bi.shape} sparsité = {sparsity(X_train_tfidf_bi):.1f} %")
print(f"HashingVectorizer : {X_train_hash.shape} sparsité = {sparsity(X_train_hash):.1f} %")
print("\n⚠️  HashingVectorizer : pas d'inverse_transform (irréversible)")


In [ ]:
# ── Figure 9 : Sparsité comparée
reps = ["Count", "TF-IDF\n(1-gram)", "TF-IDF\n(1,2-gram)", "Hashing"]
spar = [sparsity(X) for X in [X_train_count, X_train_tfidf, X_train_tfidf_bi, X_train_hash]]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(reps, spar, color=PALETTE[:4], edgecolor="white", width=0.5)
ax.set_ylim(min(spar)-2, 101); ax.set_ylabel("Sparsité (%)")
ax.set_title("Fig 9 – Sparsité des représentations textuelles", fontweight="bold")
for bar, s in zip(bars, spar):
    ax.text(bar.get_x()+bar.get_width()/2, s+0.2, f"{s:.1f}%",
            ha="center", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()


## II.3 – Construction et comparaison des pipelines textuels

In [ ]:
# Pipeline 1 : Count + Naive Bayes (baseline NLP classique)
pipe_count_nb = Pipeline([("vec", CountVectorizer(**COMMON_PARAMS)),
                           ("clf", MultinomialNB(alpha=0.1))])

# Pipeline 2 : TF-IDF + Régression Logistique (interprétable)
pipe_tfidf_lr = Pipeline([("vec", TfidfVectorizer(**COMMON_PARAMS)),
                           ("clf", LogisticRegression(max_iter=1000, C=5.0, random_state=42))])

# Pipeline 3 : TF-IDF bigrammes + LinearSVC (meilleure performance sur texte)
pipe_tfidf_svm = Pipeline([("vec", TfidfVectorizer(**COMMON_PARAMS, ngram_range=(1,2))),
                            ("clf", LinearSVC(max_iter=2000, C=1.0, random_state=42))])

# Pipeline 4 : TF-IDF + ComplementNB (robuste aux classes déséquilibrées)
pipe_tfidf_cnb = Pipeline([("vec", TfidfVectorizer(**COMMON_PARAMS)),
                            ("clf", ComplementNB(alpha=0.1))])

text_pipelines = {
    "Count + NaiveBayes"    : pipe_count_nb,
    "TF-IDF + LogReg"       : pipe_tfidf_lr,
    "TF-IDF bigram + SVM"   : pipe_tfidf_svm,
    "TF-IDF + ComplementNB" : pipe_tfidf_cnb,
}

cv_text = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
text_cv_results = {}

for name, pipe in text_pipelines.items():
    scores = cross_val_score(pipe, X_text_train, y_text_train,
                             cv=cv_text, scoring="accuracy", n_jobs=-1)
    text_cv_results[name] = {"acc_mean": scores.mean(), "acc_std": scores.std()}
    print(f"{name:<35}  Accuracy CV = {scores.mean():.4f} ± {scores.std():.4f}")


In [ ]:
# ── Figure 10 : Comparaison pipelines textuels
fig, ax = plt.subplots(figsize=(10, 4))
names_t = list(text_cv_results.keys())
means_t = [text_cv_results[n]["acc_mean"] for n in names_t]
stds_t  = [text_cv_results[n]["acc_std"]  for n in names_t]
bars = ax.bar(names_t, means_t, yerr=stds_t, capsize=7,
              color=PALETTE[:4], edgecolor="white", width=0.55)
ax.set_ylim(0, 1.05); ax.set_ylabel("Accuracy (CV 5-folds)")
ax.set_title("Fig 10 – Comparaison des pipelines textuels", fontweight="bold")
ax.tick_params(axis="x", rotation=15)
for bar, mean in zip(bars, means_t):
    ax.text(bar.get_x()+bar.get_width()/2, mean+0.01, f"{mean:.3f}",
            ha="center", fontsize=9, fontweight="bold")
plt.tight_layout(); plt.show()


## II.4 – Évaluation finale sur le jeu de test

In [ ]:
text_test_results = {}
for name, pipe in text_pipelines.items():
    pipe.fit(X_text_train, y_text_train)
    y_pred = pipe.predict(X_text_test)
    acc = accuracy_score(y_text_test, y_pred)
    f1  = f1_score(y_text_test, y_pred, average="macro")
    text_test_results[name] = {"acc": acc, "f1": f1, "y_pred": y_pred}
    print(f"\n── {name}")
    print(f"   Accuracy : {acc:.4f}   F1 macro : {f1:.4f}")
    print(classification_report(y_text_test, y_pred, target_names=CLASS_NAMES))


In [ ]:
# ── Figure 11 : Matrice de confusion du meilleur modèle
best_name = max(text_test_results, key=lambda n: text_test_results[n]["acc"])
best_pred = text_test_results[best_name]["y_pred"]
cm_text   = confusion_matrix(y_text_test, best_pred)
labels_short = [c.replace("talk.","").replace("rec.","").replace("sci.","") for c in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm_text, display_labels=labels_short).plot(ax=ax, cmap="Blues")
ax.set_title(f"Fig 11 – Matrice de confusion\n({best_name})", fontweight="bold")
plt.tight_layout(); plt.show()


## II.5 – Inspection du modèle : mots les plus discriminants

In [ ]:
# Coefficients du modèle TF-IDF + LogReg
lr_model      = pipe_tfidf_lr.named_steps["clf"]
vec_model     = pipe_tfidf_lr.named_steps["vec"]
feature_names = np.array(vec_model.get_feature_names_out())
top_n = 10

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, (ax, cls_name) in enumerate(zip(axes, CLASS_NAMES)):
    coef    = lr_model.coef_[i]
    top_idx = np.argsort(coef)[-top_n:]
    ax.barh(feature_names[top_idx], coef[top_idx], color=PALETTE[i], edgecolor="white")
    ax.set_title(cls_name.replace("talk.","").replace("rec.","").replace("sci.",""),
                 fontsize=9, fontweight="bold")
    ax.set_xlabel("Coeff. LogReg")

plt.suptitle("Fig 12 – Top-10 mots discriminants par classe (TF-IDF + LogReg)",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()


## II.6 – Analyse de l'effet des n-grammes

In [ ]:
ngram_results = {}
for ngram_range, label in [((1,1),"Unigrammes"), ((1,2),"Uni+Bigrammes"), ((1,3),"Uni+Bi+Trigrammes")]:
    pipe_ng = Pipeline([
        ("vec", TfidfVectorizer(stop_words="english", min_df=2, max_df=0.95,
                                max_features=20000, ngram_range=ngram_range)),
        ("clf", LogisticRegression(max_iter=1000, C=5.0, random_state=42))
    ])
    scores = cross_val_score(pipe_ng, X_text_train, y_text_train,
                             cv=cv_text, scoring="accuracy", n_jobs=-1)
    ngram_results[label] = scores.mean()
    print(f"{label:<25}  Accuracy CV = {scores.mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(ngram_results.keys()), list(ngram_results.values()),
       color=PALETTE[:3], edgecolor="white", width=0.5)
ax.set_ylim(min(ngram_results.values())-0.05, min(1.0, max(ngram_results.values())+0.05))
ax.set_ylabel("Accuracy (CV 5-folds)")
ax.set_title("Fig 13 – Impact des n-grammes sur la performance", fontweight="bold")
for i, val in enumerate(ngram_results.values()):
    ax.text(i, val+0.005, f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()


### 📝 Synthèse Partie II

- Le **TF-IDF** surpasse systématiquement le comptage brut en pénalisant les termes non-discriminants.
- Les représentations textuelles sont **creuses** (> 99 % de zéros) car chaque document n'utilise qu'une infime fraction du vocabulaire total (loi de Zipf).
- Les **bigrammes** capturent des contextes locaux (*"gun control"*, *"space shuttle"*) améliorant la classification.
- Le **LinearSVC** avec TF-IDF bigrammes offre le meilleur compromis performance/robustesse.
- La **Régression Logistique** est privilégiée pour l'interprétabilité via ses coefficients.


---
# Conclusion Générale 🏁

## Bilan des deux parties

| | Partie I – Titanic | Partie II – 20 Newsgroups |
|---|---|---|
| **Meilleur modèle** | RandomForest (tuned) | TF-IDF bigram + LinearSVC |
| **Accuracy** | ~82 % | ~93 % |
| **Représentation** | ColumnTransformer (num/ord/nom/bin) | TF-IDF avec bigrammes |
| **Validation** | StratifiedKFold 5 folds | StratifiedKFold 5 folds |
| **Interprétation** | Importances Gini (RF) | Coefficients LogReg |

## Apports méthodologiques
- ✅ Pipelines exempts de *data leakage* (ColumnTransformer + Pipeline)
- ✅ Validation croisée stratifiée pour des estimations robustes
- ✅ Comparaison de plusieurs stratégies de prétraitement et de modélisation
- ✅ Inspection des modèles pour l'interprétabilité

## Prolongements possibles
- Titanic : SHAP values pour l'explication locale des prédictions
- 20 Newsgroups : Word2Vec / BERT pour dépasser les limites du bag-of-words
- Gestion du déséquilibre par SMOTE ou pondération des classes
